# Sliding-Window Attention Practice

**Topic:** Efficient Attention Mechanisms &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `Q`, `K`, `V`: arrays of shape `(2, 24, 8)`.
- Window size `W = 4`: position *i* may attend to positions *j* with `i − 3 ≤ j ≤ i`.
- Scale scores by √8 before the softmax, as usual.

## Your task

With **NumPy only**:

1. Build the boolean sliding-window mask (`0 ≤ i − j < 4`).
2. Apply it to the scaled scores with `-inf`, softmax stably, and weight `V`.
3. Set `swa_out` to the `(2, 24, 8)` result.

## Evaluation

Max absolute difference from the reference must be below 1e-6.


In [ ]:
import numpy as np

rng = np.random.default_rng(461)
B, N, D = 2, 24, 8
WINDOW = 4
Q = rng.normal(0, 1.0, (B, N, D))
K = rng.normal(0, 1.0, (B, N, D))
V = rng.normal(0, 1.0, (B, N, D))
print(f"Q/K/V {Q.shape}, window {WINDOW}: produce swa_out of the same shape")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: compare against an independent reference implementation
# =====================================================================
import numpy as np

if "swa_out" not in dir():
    raise NameError("Define `swa_out` in the solution cell before running this cell.")

_i = np.arange(N)[:, None]
_j = np.arange(N)[None, :]
_mask = (_i - _j >= 0) & (_i - _j < WINDOW)
_s = Q @ K.transpose(0, 2, 1) / np.sqrt(D)
_s = np.where(_mask[None], _s, -np.inf)
_s -= _s.max(axis=-1, keepdims=True)
_w = np.exp(_s); _w /= _w.sum(axis=-1, keepdims=True)
_ref = _w @ V

_out = np.asarray(swa_out, dtype=float)
if _out.shape != _ref.shape:
    raise ValueError(f"`swa_out` has shape {_out.shape}, expected {_ref.shape}.")
_err = float(np.max(np.abs(_out - _ref)))
print("=" * 52)
print(f"Max absolute error vs reference: {_err:.2e}   (need < 1e-6)")
print(f"Each position attends to at most {WINDOW} positions. Check row 0 attends only to itself.")
print("=" * 52)
print("PASSED: efficient attention unlocked!" if _err < 1e-6 else "NOT PASSED yet: the window mask is off. Check the inequalities.")


## Solution

The sliding-window mask is just `0 <= i - j < 4` on index grids. Apply it with -inf before a stable softmax. Everything else is standard scaled dot-product attention.

```python
import numpy as np

i = np.arange(N)[:, None]
j = np.arange(N)[None, :]
mask = (i - j >= 0) & (i - j < WINDOW)

scores = Q @ K.transpose(0, 2, 1) / np.sqrt(D)
scores = np.where(mask[None], scores, -np.inf)
scores -= scores.max(axis=-1, keepdims=True)
weights = np.exp(scores)
weights /= weights.sum(axis=-1, keepdims=True)

swa_out = weights @ V
print(swa_out.shape)
```
